# PSO-Optimised CNN–LSTM for Retrospective CVD Classification (MIMIC-IV)

This notebook implements the manuscript's leakage-aware workflow: patient-level stratified 5-fold CV, training-only imputation/scaling/SMOTE, inner validation for PSO tuning, CNN–LSTM binary classification, out-of-fold predictions, metrics, plots, and saved model artifacts.

**Data requirement:** MIMIC-IV is credentialed data. This notebook does not download it or fabricate patient records. Set `DATA_CSV` to a prepared patient-level CSV exported from your authorised MIMIC-IV cohort. It must contain one row per patient/admission, a binary target column, and optional patient identifier. See the data preparation cell for requirements. Do not use this for clinical decision-making.

**Important methodology note:** if your CSV has only one row per patient and no repeated time points, the notebook treats the ordered feature vector as a short sequence for the CNN–LSTM. That is a computational representation, not genuine longitudinal modelling. For true temporal modelling, supply verified per-patient repeated measurements and build sequences without mixing patients across folds.


In [ ]:
# 1. Install dependencies (run once in Colab/Jupyter)
%pip -q install tensorflow scikit-learn imbalanced-learn pandas numpy matplotlib seaborn joblib


In [ ]:
# 2. Imports and reproducibility
import os, gc, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, Model, callbacks, regularizers
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, balanced_accuracy_score,
    matthews_corrcoef, roc_curve
)
from imblearn.over_sampling import SMOTE
import joblib

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
warnings.filterwarnings("ignore")
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 3. Configure the cohort CSV

Expected: one row per independent patient (or one row per eligible admission, with `GROUP_COL` set to patient ID so admissions from the same patient remain in one fold). `TARGET_COL` must be coded 0/1 using the manuscript's prespecified ICD-9/ICD-10 outcome definition. Exclude identifiers, post-outcome variables, and direct label leakage from predictors. Export the cohort from your authorised MIMIC-IV environment first.

In [ ]:
# EDIT THESE PATHS/COLUMN NAMES
DATA_CSV = "/content/drive/MyDrive/MIMICIV/cvd_patient_cohort.csv"
TARGET_COL = "CVD_Label"       # 0 = no CVD code, 1 = CVD code
GROUP_COL = "subject_id"       # patient identifier; set None only if each row is a unique patient
ID_COLS = ["subject_id", "hadm_id"]  # identifiers never used as predictors
OUTPUT_DIR = "/content/drive/MyDrive/MIMICIV/PSO_CNN_LSTM_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If using Google Colab, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(
        f"Cannot find {DATA_CSV}. Export your authorised MIMIC-IV cohort to this path, "
        "then update DATA_CSV. This notebook intentionally does not invent/download MIMIC-IV records."
    )

df = pd.read_csv(DATA_CSV)
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


In [ ]:
# 4. Validate labels and prepare predictors
if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL={TARGET_COL!r} not found. Available columns: {df.columns.tolist()}")
df = df.dropna(subset=[TARGET_COL]).copy()
# Accept numeric 0/1 or strings that can be converted to 0/1
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="raise").astype(int)
if not set(df[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError("Target must contain only 0 and 1.")
if df[TARGET_COL].nunique() != 2:
    raise ValueError("Both classes (0 and 1) must be present.")
if GROUP_COL is not None and GROUP_COL not in df.columns:
    raise ValueError(f"GROUP_COL={GROUP_COL!r} not found. Set GROUP_COL=None only if rows are independent.")

# Keep group/id fields aside; predictors must be numeric or convertible to numeric.
y = df[TARGET_COL].to_numpy(dtype=np.int32)
groups = df[GROUP_COL].to_numpy() if GROUP_COL is not None else None
drop_cols = set([TARGET_COL] + [c for c in ID_COLS if c in df.columns])
if GROUP_COL is not None: drop_cols.add(GROUP_COL)
X_df = df.drop(columns=list(drop_cols), errors="ignore").copy()
# One-hot encode categorical predictors; all transformations that learn data statistics
# (imputation/scaling) are fitted inside each fold below.
X_df = pd.get_dummies(X_df, dummy_na=True)
X_df = X_df.replace([np.inf, -np.inf], np.nan)
X_df = X_df.apply(pd.to_numeric, errors="coerce")
if X_df.shape[1] == 0:
    raise ValueError("No predictor columns remain after dropping target/IDs.")
feature_names = X_df.columns.astype(str).tolist()
X_raw = X_df.to_numpy(dtype=np.float32)
print("Predictor matrix:", X_raw.shape, "| class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Unique patient groups:", len(pd.unique(groups)) if groups is not None else "not supplied")


## 5. CNN–LSTM architecture and PSO

The network uses Conv1D → batch normalisation → max pooling → dropout → LSTM → dense sigmoid. To keep a notebook run feasible, the default PSO is deliberately small; increase `PSO_PARTICLES` and `PSO_ITERS` toward the manuscript's 30 particles and 50 iterations only when compute resources allow. The selected hyperparameters are chosen using inner validation F1, never the outer test fold.

In [ ]:
# Runtime controls. Start small to verify your environment.
OUTER_FOLDS = 5
INNER_VAL_SIZE = 0.20
PSO_PARTICLES = 4      # manuscript setting: 30
PSO_ITERS = 2          # manuscript setting: 50
PSO_EPOCHS = 12        # manuscript maximum: 100; early stopping applies
FINAL_EPOCHS = 100
PATIENCE = 10
VERBOSE = 0

def to_sequence(X):
    # Converts each row's ordered predictors into (steps, 1) for Conv1D/LSTM.
    # This is not a substitute for actual repeated longitudinal measurements.
    X = np.asarray(X, dtype=np.float32)
    if X.shape[1] < 2:
        X = np.concatenate([X, X], axis=1)
    return X.reshape(X.shape[0], X.shape[1], 1)

def build_model(n_features, filters=64, kernel_size=3, lstm_units=64,
                dropout=0.30, learning_rate=1e-3):
    inp = layers.Input(shape=(n_features, 1), name="clinical_feature_sequence")
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    if n_features >= 4:
        x = layers.MaxPooling1D(pool_size=2, padding="same")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units, return_sequences=False)(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=regularizers.l2(5e-4))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

def decode_particle(p):
    # p values are in [0,1]; map continuous positions to discrete choices
    return {
        "learning_rate": float(10 ** (-4 + 2*p[0])),
        "filters": [32, 64, 128][min(2, int(p[1]*3))],
        "kernel_size": [3, 5, 7][min(2, int(p[2]*3))],
        "lstm_units": [32, 64, 128][min(2, int(p[3]*3))],
        "batch_size": [16, 32, 64, 128][min(3, int(p[4]*4))],
        "dropout": float(0.10 + 0.40*p[5]),
    }

def pso_search(Xtr, ytr, Xval, yval, n_features, particles=PSO_PARTICLES, iters=PSO_ITERS):
    # SMOTE only on the training partition. Never resample validation.
    counts = np.bincount(ytr, minlength=2)
    k = min(5, int(counts.min()) - 1)
    if k >= 1 and counts.min() >= 2:
        X_bal, y_bal = SMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xtr, ytr)
    else:
        X_bal, y_bal = Xtr, ytr

    rng = np.random.default_rng(SEED)
    pos = rng.random((particles, 6))
    vel = rng.uniform(-0.1, 0.1, size=(particles, 6))
    pbest = pos.copy()
    pbest_fit = np.full(particles, -np.inf)
    gbest, gbest_fit = pos[0].copy(), -np.inf

    for iteration in range(iters):
        for i in range(particles):
            hp = decode_particle(pos[i])
            tf.keras.backend.clear_session()
            model = build_model(n_features, hp["filters"], hp["kernel_size"],
                                hp["lstm_units"], hp["dropout"], hp["learning_rate"])
            es = callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                         restore_best_weights=True, verbose=0)
            model.fit(to_sequence(X_bal), y_bal,
                      validation_data=(to_sequence(Xval), yval),
                      epochs=PSO_EPOCHS, batch_size=hp["batch_size"],
                      callbacks=[es], verbose=VERBOSE)
            prob = model.predict(to_sequence(Xval), verbose=0).ravel()
            pred = (prob >= 0.5).astype(int)
            fit = f1_score(yval, pred, zero_division=0)
            if fit > pbest_fit[i]:
                pbest_fit[i] = fit; pbest[i] = pos[i].copy()
            if fit > gbest_fit:
                gbest_fit = fit; gbest = pos[i].copy()
            del model; gc.collect()

        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        inertia = 0.9 - (0.5 * (iteration + 1) / max(1, iters))
        vel = inertia*vel + 2.0*r1*(pbest-pos) + 2.0*r2*(gbest-pos)
        pos = np.clip(pos + vel, 0, 1)
        print(f"PSO iteration {iteration+1}/{iters} | best inner-validation F1={gbest_fit:.4f}")
    return decode_particle(gbest), float(gbest_fit)


## 6. Leakage-aware patient-level cross-validation

If `GROUP_COL` is provided, outer folds are built on unique patients and all rows from a patient stay together. The notebook expects one row per patient for the primary manuscript setup; if multiple admissions per patient are supplied, grouped splitting keeps them together, but the metrics are admission-level and should be interpreted accordingly. Imputation and scaling are fitted on inner-training data for PSO, then refitted on the full outer-development data for final fold training. SMOTE is training-only.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

def make_outer_splits(X, y, groups, n_splits=5):
    if groups is None:
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        return list(splitter.split(X, y))
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    return list(splitter.split(X, y, groups))

def inner_train_val_indices(dev_idx, y, groups):
    # Return indices into the full dataset.
    if groups is None:
        tr, va = train_test_split(dev_idx, test_size=INNER_VAL_SIZE, random_state=SEED,
                                  stratify=y[dev_idx])
        return tr, va
    dev_groups = groups[dev_idx]
    gss = GroupShuffleSplit(n_splits=1, test_size=INNER_VAL_SIZE, random_state=SEED)
    tr_local, va_local = next(gss.split(dev_idx, y[dev_idx], groups=dev_groups))
    return dev_idx[tr_local], dev_idx[va_local]

def fit_preprocess(Xtrain_raw, Xother_raw):
    imp = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    Xtr = imp.fit_transform(Xtrain_raw)
    Xother = imp.transform(Xother_raw)
    Xtr = scaler.fit_transform(Xtr)
    Xother = scaler.transform(Xother)
    return Xtr.astype(np.float32), Xother.astype(np.float32), imp, scaler

outer_splits = make_outer_splits(X_raw, y, groups, OUTER_FOLDS)
oof_prob = np.full(len(y), np.nan, dtype=float)
fold_rows, fold_models = [], []

for fold, (dev_idx, test_idx) in enumerate(outer_splits, start=1):
    print(f"\n========== OUTER FOLD {fold}/{OUTER_FOLDS} ==========")
    tr_idx, val_idx = inner_train_val_indices(dev_idx, y, groups)

    # Fit preprocessing only on inner training for hyperparameter search
    Xtr, Xval, _, _ = fit_preprocess(X_raw[tr_idx], X_raw[val_idx])
    hp, best_inner_f1 = pso_search(Xtr, y[tr_idx], Xval, y[val_idx], Xtr.shape[1])
    print("Selected hyperparameters:", hp, "| inner F1:", round(best_inner_f1, 4))

    # Refit preprocessing on all outer-development data; transform untouched test
    Xdev, Xtest, imputer, scaler = fit_preprocess(X_raw[dev_idx], X_raw[test_idx])
    counts = np.bincount(y[dev_idx], minlength=2)
    k = min(5, int(counts.min()) - 1)
    if k >= 1 and counts.min() >= 2:
        Xdev_bal, ydev_bal = SMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xdev, y[dev_idx])
    else:
        Xdev_bal, ydev_bal = Xdev, y[dev_idx]

    tf.keras.backend.clear_session()
    model = build_model(Xdev.shape[1], hp["filters"], hp["kernel_size"],
                        hp["lstm_units"], hp["dropout"], hp["learning_rate"])
    # Fold-specific validation is drawn from development data only for early stopping.
    # Use the inner validation observations transformed with development-fitted preprocessing.
    Xval_final = scaler.transform(imputer.transform(X_raw[val_idx])).astype(np.float32)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                 restore_best_weights=True, verbose=0)
    model.fit(to_sequence(Xdev_bal), ydev_bal,
              validation_data=(to_sequence(Xval_final), y[val_idx]),
              epochs=FINAL_EPOCHS, batch_size=hp["batch_size"],
              callbacks=[es], verbose=VERBOSE)

    prob = model.predict(to_sequence(Xtest), verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    oof_prob[test_idx] = prob
    cm = confusion_matrix(y[test_idx], pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    row = {
        "fold": fold,
        "accuracy": accuracy_score(y[test_idx], pred),
        "precision": precision_score(y[test_idx], pred, zero_division=0),
        "sensitivity_recall": recall_score(y[test_idx], pred, zero_division=0),
        "specificity": tn/(tn+fp) if tn+fp else np.nan,
        "f1": f1_score(y[test_idx], pred, zero_division=0),
        "auroc": roc_auc_score(y[test_idx], prob) if len(np.unique(y[test_idx])) == 2 else np.nan,
        "balanced_accuracy": balanced_accuracy_score(y[test_idx], pred),
        "mcc": matthews_corrcoef(y[test_idx], pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "inner_best_f1": best_inner_f1,
        **{f"hp_{k}": v for k, v in hp.items()}
    }
    fold_rows.append(row)
    model_path = os.path.join(OUTPUT_DIR, f"cnn_lstm_fold_{fold}.keras")
    model.save(model_path)
    joblib.dump({"imputer": imputer, "scaler": scaler, "feature_names": feature_names,
                 "target_col": TARGET_COL, "group_col": GROUP_COL, "hyperparameters": hp},
                os.path.join(OUTPUT_DIR, f"preprocessing_fold_{fold}.joblib"))
    fold_models.append(model_path)
    print({k: round(v, 4) if isinstance(v, (float, np.floating)) else v
           for k,v in row.items() if k in ["accuracy","precision","sensitivity_recall","specificity","f1","auroc"]})
    del model; gc.collect()

if np.isnan(oof_prob).any():
    raise RuntimeError("Some rows have no out-of-fold predictions; check fold construction.")


In [ ]:
# 7. Aggregate metrics, fold table, pooled confusion matrix and ROC
fold_df = pd.DataFrame(fold_rows)
fold_df.to_csv(os.path.join(OUTPUT_DIR, "fold_metrics.csv"), index=False)
display(fold_df)

metric_cols = ["accuracy","precision","sensitivity_recall","specificity","f1","auroc",
               "balanced_accuracy","mcc"]
summary = pd.DataFrame({
    "metric": metric_cols,
    "mean": [fold_df[c].mean() for c in metric_cols],
    "std": [fold_df[c].std(ddof=1) for c in metric_cols]
})
summary["mean_percent_if_rate"] = summary.apply(
    lambda r: 100*r["mean"] if r["metric"] not in ["auroc","mcc"] else np.nan, axis=1)
summary.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
display(summary)

oof_pred = (oof_prob >= 0.5).astype(int)
pooled_cm = confusion_matrix(y, oof_pred, labels=[0,1])
print("Pooled out-of-fold classification report:")
print(classification_report(y, oof_pred, digits=4, zero_division=0))
print("Pooled OOF AUROC:", roc_auc_score(y, oof_prob))
print("Pooled confusion matrix [rows=true 0/1, cols=pred 0/1]:\n", pooled_cm)

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(pooled_cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No CVD","CVD"], yticklabels=["No CVD","CVD"], ax=ax)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
ax.set_title("Pooled Out-of-Fold Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pooled_confusion_matrix.png"), dpi=300)
plt.show()

fpr, tpr, _ = roc_curve(y, oof_prob)
auc = roc_auc_score(y, oof_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"PSO-CNN-LSTM (AUC={auc:.3f})")
plt.plot([0,1],[0,1],"--", label="Chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Pooled Out-of-Fold ROC Curve"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pooled_roc_curve.png"), dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plot_df = fold_df[metric_cols].melt(var_name="Metric", value_name="Score")
sns.barplot(data=plot_df, x="Metric", y="Score", errorbar="sd")
plt.xticks(rotation=30, ha="right"); plt.ylim(0,1)
plt.title("Fold-Level Performance (mean with SD)"); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fold_performance.png"), dpi=300)
plt.show()

oof = df.copy()
oof["OOF_CVD_Probability"] = oof_prob
oof["OOF_CVD_Prediction"] = oof_pred
oof.to_csv(os.path.join(OUTPUT_DIR, "out_of_fold_predictions.csv"), index=False)


In [ ]:
# 8. Save reproducibility configuration and package versions
import sklearn, imblearn
config = {
    "seed": SEED, "outer_folds": OUTER_FOLDS, "inner_validation_size": INNER_VAL_SIZE,
    "pso_particles": PSO_PARTICLES, "pso_iterations": PSO_ITERS,
    "pso_epochs": PSO_EPOCHS, "final_max_epochs": FINAL_EPOCHS,
    "early_stopping_patience": PATIENCE, "target_col": TARGET_COL,
    "group_col": GROUP_COL, "n_rows": int(len(df)), "n_predictors": int(X_raw.shape[1]),
    "class_counts": {str(k): int(v) for k,v in zip(*np.unique(y, return_counts=True))},
    "tensorflow": tf.__version__, "sklearn": sklearn.__version__, "imblearn": imblearn.__version__,
    "note": "Retrospective classification; not prospective/early prediction. Tune PSO settings upward for manuscript-scale search."
}
with open(os.path.join(OUTPUT_DIR, "run_config.json"), "w") as f:
    json.dump(config, f, indent=2)
print("Saved outputs to:", OUTPUT_DIR)
print("Files:", sorted(os.listdir(OUTPUT_DIR)))


## 9. Output files

- `fold_metrics.csv` — outer-fold metrics and selected hyperparameters
- `summary_metrics.csv` — mean and standard deviation across folds
- `out_of_fold_predictions.csv` — one untouched-fold prediction per row
- `pooled_confusion_matrix.png`, `pooled_roc_curve.png`, `fold_performance.png`
- `cnn_lstm_fold_*.keras` — fold-specific models
- `preprocessing_fold_*.joblib` — fold-specific imputer, scaler, feature names and hyperparameters
- `run_config.json` — reproducibility settings

The manuscript reports example values (accuracy 94.10%, AUROC 0.952, etc.); this notebook does **not** hard-code or guarantee those results. They must be obtained by running the code on the exact eligible cohort with its prespecified label definition and computational settings. The task is retrospective admission-level classification, not validated early prediction.